# QKSR/RSIAT ablation độc lập trên Kaggle

Notebook này chỉ phục vụ chạy thử nghiệm `A`, `B`, `G0`–`G8` trên
**ImageNet-A, ImageNet-R hoặc CIFAR224**. Repo, dataset và pretrained model
đều được đọc từ `/kaggle/input`; notebook không clone repo và không tải dataset.

Quy trình:

1. Chọn dataset, `RUN_MODE` và seeds ở cell cấu hình.
2. Chạy các cell chuẩn bị chung từ trên xuống một lần.
3. Mỗi cell experiment phía dưới tạo config và subprocess riêng. Có thể chạy
   một cell bất kỳ; không cần chạy các experiment trước nó.
4. `smoke` chỉ kiểm tra pipeline bằng 2 task/2 epoch và CA=1. `full` mới dùng
   epoch gốc để lấy kết quả nghiên cứu. Smoke và full có prefix/checkpoint khác nhau.

Khuyến nghị tài nguyên hạn chế: chạy `A`, `B`, `G0`, `G6`, `G7`, `G8` trước.
Muốn quy đóng góp từng cơ chế mới chạy thêm `G1`–`G5`. Để so sánh hợp lệ,
mọi config phải dùng cùng dataset, pretrained weights, split, class order và seeds.

## 1. Chọn dataset và chế độ smoke/full

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
PROJECT_DIR = WORK_ROOT / 'QG-RSA-ablation'
OUTPUT_ROOT = WORK_ROOT / 'qksr_ablation_runs'
CONFIG_ROOT = WORK_ROOT / 'qksr_ablation_configs'

# ===== PHẦN NGƯỜI DÙNG CẦN CHỌN =====
# Dataset: 'imageneta', 'imagenetr' hoặc 'cifar224'.
DATASET_NAME = 'imageneta'

# 'smoke': 2 task, 2 epoch/task và CA=1 để kiểm tra toàn bộ pipeline.
# 'full' : dùng nguyên epoch/CA của config nguồn, phục vụ kết quả nghiên cứu.
RUN_MODE = 'smoke'

# Smoke nên dùng một seed. Full paper-quality nên dùng ba paired seed giống
# nhau cho MỌI config, ví dụ [1993, 2024, 2025].
SEEDS = [1993]
SMOKE_TASKS = 2
FULL_RESUME = True

# 0.0 cho final protocol. Có thể dùng 0.1 trong vòng chọn cấu hình, với prefix
# và output riêng; không dùng test để chọn siêu tham số.
VAL_RATIO = 0.0
TRAIN_FRACTION = 0.8  # Chỉ dùng khi ImageNet-A/R chưa có train/test.

DATASET_SPECS = {
    'imagenetr': {
        'label': 'ImageNet-R', 'folder': 'imagenet-r',
        'config': 'adapter_imagenetr.json',
        'aliases': ('imagenet-r', 'imagenetr', 'rendition'),
        'classes': 200, 'init_cls': 20, 'increment': 20,
    },
    'imageneta': {
        'label': 'ImageNet-A', 'folder': 'imagenet-a',
        'config': 'adapter_imageneta.json',
        'aliases': ('imagenet-a', 'imageneta', 'natural-adversarial'),
        'classes': 200, 'init_cls': 20, 'increment': 20,
    },
    'cifar224': {
        'label': 'CIFAR-100 resize 224', 'folder': 'cifar-100-python',
        'config': 'adapter_cifar224.json',
        'aliases': ('cifar-100-python', 'cifar100', 'cifar-100'),
        'classes': 100, 'init_cls': 10, 'increment': 10,
    },
}
if DATASET_NAME not in DATASET_SPECS:
    raise ValueError("DATASET_NAME phải là 'imageneta', 'imagenetr' hoặc 'cifar224'.")
if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError("RUN_MODE phải là 'smoke' hoặc 'full'.")
if not SEEDS or any(not isinstance(seed, int) for seed in SEEDS):
    raise ValueError('SEEDS phải là danh sách số nguyên không rỗng.')

DATASET_SPEC = DATASET_SPECS[DATASET_NAME]
DATASET_LABEL = DATASET_SPEC['label']
DATASET_FOLDER = DATASET_SPEC['folder']
CONFIG_FILENAME = DATASET_SPEC['config']
DATASET_ALIASES = DATASET_SPEC['aliases']

# Để trống để tự tìm trong Kaggle Input.
REPO_PATH = ''
DATASET_PATH = ''
PRETRAINED_MODEL_PATH = ''

# Có thể trỏ tới qksr_ablation_runs đã upload từ session trước để full resume.
PREVIOUS_RUN_PATH = ''

assert INPUT_ROOT.is_dir(), 'Không tìm thấy /kaggle/input.'
assert 1 <= SMOKE_TASKS <= 10
assert 0.0 <= VAL_RATIO < 1.0
assert 0.0 < TRAIN_FRACTION < 1.0
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_LABEL)
print('Chế độ:', RUN_MODE)
print('Seeds:', SEEDS)

In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
print('Phiên bản PyTorch:', torch.__version__)
print('CUDA khả dụng:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Hãy bật GPU trong phần cài đặt Kaggle Notebook.')
print('GPU:', torch.cuda.get_device_name(0))

## 2. Tìm repo và pretrained model trong Kaggle Input

In [ ]:
import os
import shutil
import zipfile

def zip_chua_repo(zip_path):
    try:
        with zipfile.ZipFile(zip_path) as archive:
            names = {name.replace('\\', '/').strip('/') for name in archive.namelist()}
    except zipfile.BadZipFile:
        return False
    roots = {name[:-len('/main.py')] for name in names if name.endswith('/main.py')}
    if 'main.py' in names:
        roots.add('')
    return any(
        f'{root}/exps/{CONFIG_FILENAME}'.lstrip('/') in names
        for root in roots
    )

def tim_project_trong_thu_muc(root):
    root = Path(root)
    if not root.is_dir():
        return []
    candidates = set()
    if (root / 'main.py').is_file() and (root / 'exps' / CONFIG_FILENAME).is_file():
        candidates.add(root)
    for main_file in root.rglob('main.py'):
        project_root = main_file.parent
        if (project_root / 'exps' / CONFIG_FILENAME).is_file():
            candidates.add(project_root)
    return sorted(candidates, key=lambda path: (len(path.parts), str(path)))

if REPO_PATH:
    repo_source = Path(REPO_PATH)
    if not repo_source.exists():
        raise FileNotFoundError(f'Không tìm thấy nguồn repo: {repo_source}')
    if repo_source.is_file() and not zip_chua_repo(repo_source):
        raise ValueError(f'File không phải ZIP repo QG-RSA hợp lệ: {repo_source}')
    if repo_source.is_dir() and not tim_project_trong_thu_muc(repo_source):
        raise ValueError(f'Thư mục không chứa repo QG-RSA hợp lệ: {repo_source}')
else:
    mounted_projects = tim_project_trong_thu_muc(INPUT_ROOT)
    repo_zips = sorted(path for path in INPUT_ROOT.rglob('*.zip') if zip_chua_repo(path))
    if mounted_projects:
        # Kaggle thường tự giải nén file upload thành thư mục; ưu tiên dùng trực tiếp thư mục đó.
        if len(mounted_projects) > 1:
            raise RuntimeError(
                'Tìm thấy nhiều thư mục repo. Hãy điền REPO_PATH bằng một trong các đường dẫn:\n'
                + '\n'.join(map(str, mounted_projects))
            )
        repo_source = mounted_projects[0]
    elif repo_zips:
        if len(repo_zips) > 1:
            raise RuntimeError(
                'Tìm thấy nhiều ZIP repo. Hãy điền REPO_PATH bằng một trong các đường dẫn:\n'
                + '\n'.join(map(str, repo_zips))
            )
        repo_source = repo_zips[0]
    else:
        raise FileNotFoundError(
            'Không tìm thấy file ZIP hoặc thư mục repo trong /kaggle/input. '            'Hãy kiểm tra Input hoặc điền REPO_PATH.'
        )

if repo_source.is_file():
    extract_dir = WORK_ROOT / 'qg_rsa_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    extract_root = extract_dir.resolve()
    with zipfile.ZipFile(repo_source) as archive:
        for member in archive.infolist():
            destination = (extract_root / member.filename).resolve()
            if destination != extract_root and extract_root not in destination.parents:
                raise ValueError(f'Đường dẫn không an toàn trong ZIP: {member.filename}')
        archive.extractall(extract_root)
    project_candidates = tim_project_trong_thu_muc(extract_root)
else:
    project_candidates = tim_project_trong_thu_muc(repo_source)

if len(project_candidates) != 1:
    raise RuntimeError(f'Nguồn phải chứa đúng một project QG-RSA, tìm thấy {len(project_candidates)}.')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(project_candidates[0], PROJECT_DIR)

if PREVIOUS_RUN_PATH:
    previous_run = Path(PREVIOUS_RUN_PATH)
    if not previous_run.is_dir():
        raise FileNotFoundError(f'Không tìm thấy lần chạy trước: {previous_run}')
    shutil.copytree(previous_run, OUTPUT_ROOT, dirs_exist_ok=True)
    print('Đã khôi phục checkpoint từ:', previous_run)

os.chdir(PROJECT_DIR)
print('Nguồn repo:', repo_source)
print('Thư mục project:', PROJECT_DIR)
print('Thư mục kết quả:', OUTPUT_ROOT)

In [ ]:
# Không cài lại torch/torchvision để giữ bộ CUDA tương thích có sẵn trên Kaggle.
%pip install -q --upgrade-strategy only-if-needed -r requirements-colab.txt

import timm
import torchvision
print('torchvision:', torchvision.__version__)
print('timm:', timm.__version__)

# Dùng pretrained model đã upload trong Input, không gọi Hugging Face.
if PRETRAINED_MODEL_PATH:
    pretrained_path = Path(PRETRAINED_MODEL_PATH)
    if not pretrained_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy pretrained model: {pretrained_path}')
else:
    model_candidates = sorted(INPUT_ROOT.rglob('model.safetensors'))
    if len(model_candidates) != 1:
        raise RuntimeError(
            f'Cần đúng một model.safetensors trong Input, tìm thấy {len(model_candidates)}. '
            'Hãy điền PRETRAINED_MODEL_PATH nếu có nhiều file.'
        )
    pretrained_path = model_candidates[0]

from safetensors import safe_open
with safe_open(str(pretrained_path), framework='pt', device='cpu') as checkpoint:
    pretrained_keys = set(checkpoint.keys())
required_keys = {'patch_embed.proj.weight', 'blocks.0.attn.qkv.weight'}
missing_keys = required_keys - pretrained_keys
if missing_keys:
    raise ValueError(f'Checkpoint không đúng ViT-B/16 của timm; thiếu key: {sorted(missing_keys)}')

adapter_source = PROJECT_DIR / 'network' / 'vision_transformer_adapter.py'
adapter_text = adapter_source.read_text(encoding='utf-8')
online_line = '    checkpoint_model = timm.create_model("vit_base_patch16_224_in21k", pretrained=True, num_classes=0)'
local_block = f'''    checkpoint_model = timm.create_model(
        "vit_base_patch16_224.augreg_in21k", pretrained=False, num_classes=0
    )
    from safetensors.torch import load_file
    local_state_dict = load_file({str(pretrained_path)!r})
    local_state_dict.pop("head.weight", None)
    local_state_dict.pop("head.bias", None)
    local_load = checkpoint_model.load_state_dict(local_state_dict, strict=False)
    if local_load.missing_keys or local_load.unexpected_keys:
        raise ValueError(f'Checkpoint local không khớp backbone: {{local_load}}')'''
if online_line in adapter_text:
    adapter_source.write_text(adapter_text.replace(online_line, local_block, 1), encoding='utf-8')
    print('Đã cấu hình pretrained model local:', pretrained_path)
elif 'local_state_dict = load_file(' in adapter_text:
    print('Source đã dùng pretrained model local.')
else:
    raise RuntimeError('Không tìm thấy đoạn nạp pretrained model cần thay trong source.')

subprocess.run([
    'python', '-m', 'py_compile', 'main.py', 'trainer.py',
    'models/base.py', 'models/RSIAT_adapter.py',
    'utils/quantum_kernel.py', 'utils/qksr_statistics.py'
], check=True)
print('Kiểm tra cú pháp mã nguồn: đạt.')

## 3. Chuẩn bị dataset local, không tải mạng

In [ ]:
import random
import shutil
import tarfile
import zipfile

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def thu_muc_lop(root):
    root = Path(root)
    if not root.is_dir():
        return []
    return sorted(
        directory for directory in root.iterdir()
        if directory.is_dir() and any(
            file.is_file() and file.suffix.lower() in IMAGE_SUFFIXES
            for file in directory.iterdir()
        )
    )

def loai_cau_truc_imagenet(root):
    root = Path(root)
    train_classes, test_classes = thu_muc_lop(root / 'train'), thu_muc_lop(root / 'test')
    expected = DATASET_SPEC['classes']
    if len(train_classes) == expected and len(test_classes) == expected:
        if [p.name for p in train_classes] == [p.name for p in test_classes]:
            return 'đã chia train/test'
    if len(thu_muc_lop(root)) == expected:
        return 'chưa chia'
    return None

def cifar_root_hop_le(root):
    root = Path(root)
    return root.is_dir() and all((root / name).is_file() for name in ('train', 'test', 'meta'))

if DATASET_NAME == 'cifar224':
    candidates = []
    if DATASET_PATH:
        candidate = Path(DATASET_PATH)
        candidates = [candidate / 'cifar-100-python' if cifar_root_hop_le(candidate / 'cifar-100-python') else candidate]
    else:
        candidates = [path for path in INPUT_ROOT.rglob('cifar-100-python') if cifar_root_hop_le(path)]
    candidates = sorted(set(path.resolve() for path in candidates if cifar_root_hop_le(path)))
    if len(candidates) != 1:
        raise RuntimeError(
            f'Cần đúng một thư mục cifar-100-python đã giải nén trong Input, tìm thấy {len(candidates)}. '
            'Thư mục phải chứa train, test và meta; điền DATASET_PATH nếu cần.'
        )
    prepared_root = candidates[0]
    repo_data = PROJECT_DIR / 'data' / 'datasets' / 'cifar-100-python'
    repo_data.parent.mkdir(parents=True, exist_ok=True)
    if repo_data.is_symlink() or repo_data.is_file():
        repo_data.unlink()
    elif repo_data.exists():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(prepared_root, target_is_directory=True)
    print('CIFAR-100 local:', prepared_root)
    print('torchvision sẽ kiểm tra file local; notebook không tải dataset.')
else:
    if DATASET_PATH:
        source_root = Path(DATASET_PATH)
        if not source_root.is_relative_to(INPUT_ROOT):
            raise ValueError('DATASET_PATH phải nằm trong /kaggle/input.')
        if loai_cau_truc_imagenet(source_root) is None:
            raise ValueError(f'Không nhận ra cấu trúc {DATASET_LABEL}: {source_root}')
        candidates = [source_root]
    else:
        split_candidates, flat_candidates = [], []
        for directory in INPUT_ROOT.rglob('*'):
            if not directory.is_dir():
                continue
            normalized = str(directory).lower().replace('_', '-')
            if not any(alias in normalized for alias in DATASET_ALIASES):
                continue
            layout = loai_cau_truc_imagenet(directory)
            if layout == 'đã chia train/test':
                split_candidates.append(directory)
            elif layout == 'chưa chia':
                flat_candidates.append(directory)
        candidates = split_candidates if split_candidates else flat_candidates
        candidates = sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))
    if len(candidates) != 1:
        raise RuntimeError(
            f'Cần đúng một thư mục {DATASET_LABEL} trong Input, tìm thấy {len(candidates)}. '
            'Điền DATASET_PATH nếu có nhiều ứng viên.'
        )
    source_root = candidates[0]
    layout = loai_cau_truc_imagenet(source_root)
    if layout == 'đã chia train/test':
        prepared_root = source_root
    else:
        prepared_root = WORK_ROOT / f'{DATASET_FOLDER}-ablation-prepared'
        if prepared_root.exists():
            shutil.rmtree(prepared_root)
        rng = random.Random(SEEDS[0])
        for class_dir in thu_muc_lop(source_root):
            images = sorted(path for path in class_dir.iterdir()
                            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
            if len(images) < 2:
                raise ValueError(f'Lớp {class_dir.name} có ít hơn 2 ảnh.')
            rng.shuffle(images)
            cut = min(max(1, int(len(images) * TRAIN_FRACTION)), len(images) - 1)
            for split, selected in (('train', images[:cut]), ('test', images[cut:])):
                destination = prepared_root / split / class_dir.name
                destination.mkdir(parents=True, exist_ok=True)
                for image in selected:
                    (destination / image.name).symlink_to(image.resolve())
    train_classes = thu_muc_lop(prepared_root / 'train')
    test_classes = thu_muc_lop(prepared_root / 'test')
    if len(train_classes) != DATASET_SPEC['classes'] or [p.name for p in train_classes] != [p.name for p in test_classes]:
        raise ValueError('Số/tên lớp train và test không hợp lệ.')
    repo_data = PROJECT_DIR / 'data' / 'datasets' / DATASET_FOLDER
    repo_data.parent.mkdir(parents=True, exist_ok=True)
    if repo_data.is_symlink() or repo_data.is_file():
        repo_data.unlink()
    elif repo_data.exists():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(Path(prepared_root).resolve(), target_is_directory=True)
    train_count = sum(sum(1 for p in directory.iterdir() if p.is_file()) for directory in train_classes)
    test_count = sum(sum(1 for p in directory.iterdir() if p.is_file()) for directory in test_classes)
    print(f'{DATASET_LABEL}: {len(train_classes)} lớp | train={train_count:,} | test={test_count:,}')

print('Liên kết dataset:', repo_data, '->', repo_data.resolve())

## 4. Test source và kiểm tra protocol 10 task

In [ ]:
import sys

subprocess.run([
    sys.executable, '-m', 'unittest', 'discover',
    '-s', str(PROJECT_DIR / 'tests'), '-p', 'test_*.py', '-q',
], cwd=PROJECT_DIR, check=True)

from data.data_manager import DataManager
from models.RSIAT_adapter import Learner
from scripts.generate_qksr_ablation_configs import VARIANTS

EXPERIMENTS = [
    'A_rsiat', 'B_qksr',
    'G0_paired_current_margin', 'G1_signed_projector', 'G2_reset_projector',
    'G3_detached_prototypes', 'G4_relation_retention',
    'G5_mean_cov_transport', 'G6_cov_shrinkage',
    'G7_retention_rbf', 'G8_rsiat_shared_retention',
]
missing = set(EXPERIMENTS) - set(VARIANTS)
if missing:
    raise RuntimeError(f'Source ZIP chưa có các ablation mới: {sorted(missing)}')
if not hasattr(Learner, '_transport_old_statistics'):
    raise RuntimeError('Source ZIP chưa có bản retention mới.')

probe = DataManager(
    DATASET_NAME, True, SEEDS[0],
    DATASET_SPEC['init_cls'], DATASET_SPEC['increment'],
)
if probe.nb_tasks != 10:
    raise RuntimeError(f'Protocol phải có 10 task, nhận được {probe.nb_tasks}.')
print('Preflight đạt | tasks:', probe.nb_tasks, '| class order đầu:', probe._class_order[:20])

## 5. Hàm tạo/chạy từng cấu hình độc lập

In [ ]:
import copy
import json
import sys
from scripts.generate_qksr_ablation_configs import VARIANTS

QUANTUM_DEFAULTS = {
    'use_quantum_kernel_base': True, 'use_quantum_kernel_inc': True,
    'q_kernel_type': 'pqk', 'q_kernel_order': 1, 'q_order2_weight': 1.0,
    'q_num_qubits': 8, 'q_num_layers': 2, 'q_reupload': False,
    'q_dtype': 'float32', 'q_gamma_mode': 'bounded_learned',
    'q_calib_samples': 512, 'q_init_seed': 1234, 'q_metric_lr_mult': 0.1,
    'q_inc_train_mode': 'frozen', 'q_inc_pair': 'old_proj', 'inc_loss_mode': 'mean',
    'q_inc_weight': 1.0, 'q_inc_warmup_epochs': 0,
    'ssca_feature_mode': 'legacy', 'rs_margin_q': 0.5,
    'rs_margin_q_mode': 'fixed', 'rs_margin_quantile': 0.9, 'rs_margin_inc': 0.3,
}

def build_experiment_config(experiment_name):
    if experiment_name not in EXPERIMENTS:
        raise ValueError(f'Experiment không được notebook này hỗ trợ: {experiment_name}')
    source_path = PROJECT_DIR / 'exps' / CONFIG_FILENAME
    source = json.loads(source_path.read_text(encoding='utf-8'))
    config = {**QUANTUM_DEFAULTS, **source}
    config.update(copy.deepcopy(VARIANTS[experiment_name]))

    mode_suffix = 'smoke' if RUN_MODE == 'smoke' else 'full'
    validation_suffix = f'_val{VAL_RATIO:g}' if VAL_RATIO else ''
    config.update({
        'prefix': f'{DATASET_NAME}_{experiment_name}_{mode_suffix}{validation_suffix}',
        'dataset': DATASET_NAME,
        'seed': list(SEEDS),
        'device': ['0'],
        'output_root': str(OUTPUT_ROOT),
        'isolate_runs': True,
        'resume': RUN_MODE == 'full' and FULL_RESUME,
        'val_ratio': VAL_RATIO,
        'num_workers': 2,
        'stats_num_workers': 2,
        'pin_memory': True,
        'persistent_workers': True,
    })
    if experiment_name.startswith('G'):
        config['keep_last_checkpoint'] = False
    if RUN_MODE == 'smoke':
        config.update({
            'resume': False,
            'max_tasks_per_run': SMOKE_TASKS,
            'init_epochs': 2,
            'inc_epochs': 2,
            'ca_epochs': 1,
            'batch_size': min(32, int(config['batch_size'])),
        })
    else:
        config.pop('max_tasks_per_run', None)
    return config

def run_experiment(experiment_name):
    """Tạo config mới và chạy đúng một experiment; không phụ thuộc cell run khác."""
    config = build_experiment_config(experiment_name)
    config_path = CONFIG_ROOT / f"{config['prefix']}.json"
    config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    print('=' * 80)
    print('EXPERIMENT:', experiment_name)
    print('MODE:', RUN_MODE.upper())
    if RUN_MODE == 'smoke':
        print(f"SMOKE: {config['max_tasks_per_run']} task, init/inc=2 epoch, CA=1. Không dùng accuracy để báo cáo.")
    else:
        print('FULL: dùng toàn bộ epoch/CA của config nguồn. Có thể resume checkpoint cùng prefix/seed.')
    print('Dataset:', DATASET_NAME, '| Seeds:', config['seed'])
    print('Config:', config_path)
    command = [sys.executable, '-u', 'main.py', '--config', str(config_path)]
    print('Lệnh:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    print('HOÀN THÀNH:', experiment_name)
    return config_path

print('Đã nạp runner. Chỉ chạy các cell experiment bạn cần ở bên dưới.')

## 6. Các cell experiment

Mỗi cell dưới đây chạy **đúng một cấu hình** theo `RUN_MODE` đã chọn.
Trong smoke, không so accuracy với paper. Trước full, đổi `RUN_MODE='full'`,
chọn cùng danh sách `SEEDS`, rồi chạy lại từ cell cấu hình đến runner.

### 1. `A_rsiat` — RSIAT gốc

Baseline không quantum, không retention mới.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('A_rsiat')

### 2. `B_qksr` — QKSR legacy

QKSR hiện tại trước chuỗi cải tiến retention.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('B_qksr')

### 3. `G0_paired_current_margin` — QKSR current-margin + paired SSCA

Điểm bắt đầu công bằng của chuỗi retention.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G0_paired_current_margin')

### 4. `G1_signed_projector` — Thêm projector residual có dấu

Tách hạn chế Sigmoid một chiều và khởi tạo identity.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G1_signed_projector')

### 5. `G2_reset_projector` — Reset projector mỗi task

Kiểm tra bias khi tái sử dụng ánh xạ drift cũ.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G2_reset_projector')

### 6. `G3_detached_prototypes` — Detach prototype khỏi repulsion

Ngăn prototype tự di chuyển để né loss QKSR.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G3_detached_prototypes')

### 7. `G4_relation_retention` — Thêm cosine relation retention

Neo quan hệ prototype-instance theo hình học classifier.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G4_relation_retention')

### 8. `G5_mean_cov_transport` — Transport mean và covariance

Ánh xạ ridge có holdout/support guard.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G5_mean_cov_transport')

### 9. `G6_cov_shrinkage` — Full retention PQK

Ứng viên QKSR đầy đủ, thêm covariance shrinkage.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G6_cov_shrinkage')

### 10. `G7_retention_rbf` — Full retention RBF

Đối chứng PQK/RBF với cùng objective và retention.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G7_retention_rbf')

### 11. `G8_rsiat_shared_retention` — RSIAT + shared retention

Đối chứng không quantum nhưng giữ các sửa chung.

- `smoke`: kiểm tra code/pipeline, không báo cáo accuracy.
- `full`: thực nghiệm hoàn chỉnh; dùng cùng paired seeds với mọi cấu hình.

In [ ]:
run_experiment('G8_rsiat_shared_retention')

## 7. Tổng hợp đường dẫn kết quả

Cell này không huấn luyện. Nó liệt kê config/log/checkpoint đã tạo. Checkpoint
không được đưa vào ZIP metadata vì rất lớn; cần lưu toàn bộ
`/kaggle/working/qksr_ablation_runs` thành Kaggle Dataset nếu muốn resume.

In [ ]:
import shutil

configs = sorted(CONFIG_ROOT.glob('*.json'))
logs = sorted((OUTPUT_ROOT / 'logs' / 'adapter' / DATASET_NAME).rglob('*.log'))
checkpoints = sorted((OUTPUT_ROOT / 'ckpt').rglob('task_*.pkl'))
print('Configs:', len(configs))
print('Logs:', len(logs))
print('Checkpoints:', len(checkpoints))
for path in logs[-20:]:
    print('LOG:', path)

bundle = WORK_ROOT / f'{DATASET_NAME}_ablation_metadata'
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
for path in configs:
    shutil.copy2(path, bundle / path.name)
for index, path in enumerate(logs):
    shutil.copy2(path, bundle / f'{index:03d}_{path.name}')
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('ZIP config + log:', archive)
print('Thư mục cần lưu để resume:', OUTPUT_ROOT)